In [ ]:
# Hyperparameter Tuning
  # Number of Hidden Layer
  # Number of nerons per layer
  # Number of Epochs
  # Optimizer
  # Learning Rate
  # Batchsize
  # Dropout Rate
  # Weight Decay(lamda)


In [1]:
# import Required Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [3]:
# Check GPU availability
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Available Device is: {device}')

Available Device is: cuda


In [4]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Extract Features values and Target values
X=df.iloc[:, 1:].to_numpy()
y=df.iloc[:, 0].to_numpy()

In [6]:
# Split the Dataset into train and testset
X_train,X_test, y_train,y_test=train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [7]:
# Scaling the features
X_train=X_train/255.0
X_test=X_test/255.0

In [8]:
# Create CustomDataset Class
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features=torch.tensor(features, dtype=torch.float32)
        self.labels=torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
# Create train dataset object
train_dataset=CustomDataset(X_train, y_train)

In [10]:
# Create test Dataset object
test_dataset=CustomDataset(X_test, y_test)

In [11]:
# Define Neural Network Class

class MyNN(nn.Module):

    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()

        layers=[]
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim=neurons_per_layer

        layers.append(nn.Linear(input_dim, output_dim))
        self.model=nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [12]:
# Create Objective Function
def objective(trial):
  # Hyperparameter value from the search space
  num_hidden_layers=trial.suggest_int('num_hidden_layers', 1, 5)
  neurons_per_layer=trial.suggest_int('neurons_per_layer', 8, 128, step=8)
  epochs=trial.suggest_int('epochs', 10, 50, step=10)
  learning_rate=trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
  dropout_rate=trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
  batch_size=trial.suggest_categorical('batch_size', [16, 32, 64, 128])
  optimizer_name=trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
  weight_decay=trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

  # Create train and test dataloader
  train_loader=DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
  test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

  # Model Init
  input_dim=784
  output_dim=10

  model=MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model=model.to(device)

  # Loss Function
  criterion=nn.CrossEntropyLoss()

  if optimizer_name=='Adam':
    optimizer=optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name=='SGD':
    optimizer=optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer=optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # Training Loop
  for epochs in range(epochs):
    for batch_features, batch_labels in train_loader:

      # Move data to GPU
      batch_features, batch_labels=batch_features.to(device), batch_labels.to(device)

      # Forward pass
      outputs=model(batch_features)

      # Calculate loss
      loss=criterion(outputs, batch_labels)

      # Back pass
      optimizer.zero_grad()
      loss.backward()

      # Update grads
      optimizer.step()


  # Evaluation

  model.eval()
  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy


In [13]:
# Install optuna
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 29.6 MB/s eta 0:00:00


In [14]:
# Create optuna Study
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-09-26 10:13:09,327] A new study created in memory with name: no-name-baa37e41-38c1-43b1-a39d-9e514c795046


In [15]:
# Create trial
study.optimize(objective, n_trials=10)

[I 2026-09-26 10:14:15,545] Trial 0 finished with value: 0.5610833333333334 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 8, 'epochs': 10, 'learning_rate': 0.06386840351888337, 'dropout_rate': 0.1, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 0.0006847301093027447}. Best is trial 0 with value: 0.5610833333333334.
[I 2026-09-26 10:15:41,233] Trial 1 finished with value: 0.892 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 120, 'epochs': 40, 'learning_rate': 0.07909353655156476, 'dropout_rate': 0.30000000000000004, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.00010106048749990871}. Best is trial 1 with value: 0.892.
[I 2026-09-26 10:16:29,291] Trial 2 finished with value: 0.7224166666666667 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 104, 'epochs': 20, 'learning_rate': 5.509342985206298e-05, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.0008266125553932614}. Best is trial 1 with value:

In [ ]:
# Decrease the accuracy difference between test and train dataset after applying optimization

In [16]:
# Finding Best Value
study.best_value

0.892

In [17]:
# Finding Best Parameters
study.best_params

{'num_hidden_layers': 3,
 'neurons_per_layer': 120,
 'epochs': 40,
 'learning_rate': 0.07909353655156476,
 'dropout_rate': 0.30000000000000004,
 'batch_size': 64,
 'optimizer': 'SGD',
 'weight_decay': 0.00010106048749990871}